In [49]:
# import
import pandas as pd
import os
import numpy as np

In [50]:
def safe_read_csv(filepath: str) -> pd.DataFrame:
    """
    여러 인코딩을 시도하여 CSV 파일을 안전하게 읽는 함수

    Parameters:
        filepath (str): CSV 파일 경로

    Returns:
        pd.DataFrame: 성공적으로 로드된 DataFrame
    """
    # 사용할 인코딩 후보 리스트
    encodings = ["cp949", "utf-8-sig", "utf-8", "euc-kr"]
    last_error = None  # 마지막으로 발생한 오류 저장

    # 후보 인코딩을 순서대로 시도
    for enc in encodings:
        try:
            # 주어진 인코딩으로 CSV 읽기
            df = pd.read_csv(filepath, encoding=enc)

            # 읽은 파일이 비어 있으면 오류 발생
            if df.empty:
                raise ValueError("CSV 파일이 비어 있습니다.")

            # 정상적으로 읽었으면 DataFrame 반환
            return df

        except Exception as e:
            # 실패하면 오류 기록 후 다음 인코딩 시도
            last_error = e
            continue

    # 모든 인코딩 시도 후에도 실패한 경우 예외 발생
    raise ValueError(f"CSV 로드 실패: {last_error}")


In [51]:
# 시도와 시군구 컬럼을 결합하여 하나의 'SGG_NAME' 컬럼을 생성
def combine_region_columns(
    df: pd.DataFrame,
    col_sido: str = "시군구별(1)",
    col_sigungu: str = "시군구별(2)",
    new_col: str = "시군구별"
) -> pd.DataFrame:
    """
    시도와 시군구 컬럼을 결합하여 하나의 'SGG_NAME' 컬럼을 생성

    Parameters:
        df (pd.DataFrame): 원본 DataFrame
        col_sido (str): 시도 컬럼명 (기본값: '시군구별(1)')
        col_sigungu (str): 시군구 컬럼명 (기본값: '시군구별(2)')
        new_col (str): 생성될 컬럼명 (기본값: '시군구_전체')

    Returns:
        pd.DataFrame: 시군구_전체 컬럼이 추가된 DataFrame
    """
    # 0. 소계 제거 (단, 세종특별자치시 소계는 보존)
    df = df[~((df["시군구별(2)"] == "소계") & (df["시군구별(1)"] != "세종특별자치시"))]
    # df = df[~((df["시군구별(2)"] == "소계"))]
    
    # 1. 시군구 결합
    df[new_col] = df[col_sido].str.strip() + " " + df[col_sigungu].str.strip()

    # 2. 기존 컬럼 제거
    df.drop(columns=[col_sido, col_sigungu], inplace=True)

    # 3. 새 컬럼을 가장 왼쪽으로 이동
    cols = [new_col] + [col for col in df.columns if col != new_col]
    df = df[cols]


    return df


In [ ]:
# 시/도 이름 정규화
def normalize_sido_names(df: pd.DataFrame, column: str, col_cnt: int) -> pd.DataFrame:
    """
    주어진 컬럼에서 시도 약칭을 공통된 명칭으로 표준화, 시군구 컬럼명 통일('SGG_NAME')

    Parameters:
        df (pd.DataFrame): 대상 DataFrame
        column (str): 변환할 컬럼명 ("시군구별(1)" 또는 "시군구별")
        col_cnt (int): 시군구 컬럼 개수 (1 또는 2)

    Returns:
        pd.DataFrame: 치환된 DataFrame
    """
    
     # 1) 시도 약칭/옛 표기 → 정식 명칭 매핑 사전
    mapping = {
        "서울": "서울특별시", "인천": "인천광역시", "경기": "경기도",
        "부산": "부산광역시", "대구": "대구광역시", "광주": "광주광역시",
        "대전": "대전광역시", "울산": "울산광역시", "세종": "세종특별자치시",
        "강원": "강원특별자치도", "충북": "충청북도", "충남": "충청남도",
        "전북": "전북특별자치도", "전남": "전라남도", "경북": "경상북도",
        "경남": "경상남도", "제주": "제주특별자치도",
        "전라북도": "전북특별자치도", "강원도": "강원특별자치도"
}

    # 2) 대상 컬럼 존재 확인
    if column not in df.columns:
        raise KeyError(f"'{column}' 컬럼없음")

    # 3) 개별 값 변환 함수 정의
    def convert_region_name(value):
        value = str(value).strip()
        for short, full in mapping.items():
            # 3-1) 이미 정식 명칭이면 그대로 반환
            if value.startswith(full):
                return value
            # 3-2) "강원도", "충북도" 같이 도 단위 매칭
            if value.startswith(short + "도"):
                return value.replace(short + "도", full, 1)
            # 3-3) 약칭일 때 정식 명칭으로 치환
            if value.startswith(short):
                return f"{full} {value[len(short):].strip()}"
        return value

    # 4) 시도명 변환 실행 & 컬럼명 'SGG_NAME'으로 변경
    if col_cnt == 1:
        df[column] = df[column].apply(convert_region_name)
        df.rename(columns={column: "SGG_NAME"}, inplace=True)
    else:
        df[column] = df[column].apply(convert_region_name)
        df.rename(columns={"시군구별": "SGG_NAME"}, inplace=True)
    
    # 5) 예외 처리: 잘못된 표기 수정
    df["SGG_NAME"] = df["SGG_NAME"].replace({
        "세종특별자치시 세종시": "세종특별자치시",
        "세종특별자치시 세종특별자치시": "세종특별자치시",
        "경상남도 창원시(통합)": "경상남도 창원시",
        "부산광역시 진구": "부산진구",
        "제주특별자치도 제주특별자치도 시": "제주특별자치도 제주시",
    })
    
    # 6) 명백히 잘못된 중복 표기 제거
    df = df[df['SGG_NAME'] != '광주광역시 광주광역시']
    df = df[df['SGG_NAME'] != '부산광역시 부산광역시']

    # ⑧ 필요 시 중복 처리/정렬 (현재는 주석 처리)
    # df = df.groupby("SGG_NAME", as_index=False).first()
    # df = df.sort_values(by="SGG_NAME").reset_index(drop=True)
    
    # 디버깅용 (csv 저장)
    df.to_csv("./test.csv", encoding="utf-8-sig", index=False)
    return df

In [53]:
# 파일 내 첫 번째 컬럼명을 확인하고, 시군구 컬럼(법정동 정보)을 표준화, 정규화하는 함수
def integration_sgg_col(
    filepath: str
)-> pd.DataFrame:
    """
    주어진 CSV 파일의 첫 번째 컬럼명을 확인하여
    시군구 컬럼을 표준화·정규화한 뒤 result.csv로 저장

    Parameters:
        filepath (str): 입력 CSV 파일 경로

    Returns:
        pd.DataFrame: 시군구 컬럼 정제 및 표준화가 완료된 DataFrame
    """
    # 1) 인코딩 오류 방지하며 파일 불러오기
    df = safe_read_csv(filepath)
    first_col = df.columns[0] # 첫 번째 컬럼명

    # 2) 케이스 분기 
    if first_col == "시군구별(1)": # 법정동 컬럼 2개
        df = combine_region_columns(df) # 시군구 컬럼 2개 하나로 합치기
        df = normalize_sido_names(df, column="시군구별", col_cnt=2) # 시도 치환
        df = merge_subdistricts_to_city(df, region_col="SGG_NAME") # 하위 행정구역 통합
    elif first_col =="시군구별": # 법정동 컬럼 1개
        df = normalize_sido_names(df, column=first_col, col_cnt=1) # 시도 치환
        df = create_full_region_column(df, column="SGG_NAME") # 시군구_전체 컬럼 생성
    else: # 둘다 아닐때 데이터 초기 전처리 잘못되었으므로 오류 발생 내용 추가
        raise ValueError(
            f"지원하지 않는 첫 컬럼명: '{first_col}'. 허용: '시군구별(1)', '시군구별'"
        )
    
    df = merge_gunwigun(df, region_col="SGG_NAME") # 군위군 데이터 통합
    df = remove_target_regions(df, region_col="SGG_NAME") # 특례시 하위 행정구역 제거
    df = remove_only_sido_rows(df, region_col="SGG_NAME") # 시도만 있는 행 제거 
    df = remove_seoul_gyeonggi_incheon(df)
    df = check_missing_regions(df, region_col="SGG_NAME")
    validate_and_save_dataframe(df)
    
    # 이거 함수 구현
    df.to_csv("./result.csv", encoding="utf-8-sig", index=False)
    return df

In [54]:
def data_cleansing_folder(
    folder_path: str, 
    output_folder: str
)-> None:
    """
    모든 하위 폴더의 .csv 파일을 읽어 정제 후 output_folder에 저장

    Parameters:
        folder_path (str): 입력 CSV 파일들이 들어있는 최상위 폴더
        output_folder (str): 정제된 CSV를 저장할 최상위 폴더
    """
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".csv"):  # CSV 파일만 처리
                try:
                    # CSV 읽어서 시군구 컬럼 정제 (사용자 정의 함수)
                    df = integration_sgg_col(os.path.join(root, file))

                    # 원본 폴더 구조 보존: folder_path 기준 상대경로 계산
                    rel = os.path.relpath(root, folder_path)
                    save_dir = os.path.join(output_folder, rel)
                    os.makedirs(save_dir, exist_ok=True)  # 저장할 폴더 생성

                    # 저장 파일명: "_정리.csv" 붙이기
                    save_path = os.path.join(save_dir, file.replace(".csv", "_정리.csv"))

                    # 정제된 CSV 저장
                    df.to_csv(save_path, index=False, encoding="utf-8-sig")
                    print(f"✅ 저장 완료: {save_path}")

                except Exception as e:
                    # 오류 발생 시 로그 출력 (다른 파일 처리는 계속 진행)
                    print(f"❌ 오류 발생 {file}: {e}")

In [55]:
# 수정 필요
input_path = "../../data/01-1_data-cleansing"
# 수정 필요
output_path = "../../data/01-2_data-cleansing"


# 나중에 하나
# 폴더 안 파일들 전체 데이터 정제
data_cleansing_folder(input_path, output_path)

❌ 오류 발생 교원_1인당_학생수_컬럼명변경.csv: name 'create_full_region_column' is not defined
❌ 오류 발생 대학교 진학률_컬럼명변경.csv: name 'merge_subdistricts_to_city' is not defined
❌ 오류 발생 대학교_교원수_컬럼명변경.csv: name 'create_full_region_column' is not defined
❌ 오류 발생 대학교_수_컬럼명변경.csv: name 'create_full_region_column' is not defined
❌ 오류 발생 유치원_교원수_컬럼명변경.csv: name 'create_full_region_column' is not defined
❌ 오류 발생 유치원_수_컬럼명변경.csv: name 'create_full_region_column' is not defined
❌ 오류 발생 유치원_원아수_컬럼명변경.csv: name 'create_full_region_column' is not defined
❌ 오류 발생 인구_천명당_사설학원수_컬럼명변경.csv: name 'create_full_region_column' is not defined
❌ 오류 발생 초등학교_교원수_컬럼명변경.csv: name 'create_full_region_column' is not defined
❌ 오류 발생 초등학교_학생수_컬럼명변경.csv: name 'create_full_region_column' is not defined
❌ 오류 발생 학급당_학생수_컬럼명변경.csv: name 'create_full_region_column' is not defined
❌ 오류 발생 남녀성비_시도_시_군_구__20250729160623.csv: name 'merge_subdistricts_to_city' is not defined
❌ 오류 발생 재정자립도_컬럼명변경.csv: name 'merge_subdistricts_to_city' is not defined


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_2920\349334214.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[new_col] = df[col_sido].str.strip() + " " + df[col_sigungu].str.strip()
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_2920\349334214.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns=[col_sido, col_sigungu], inplace=True)
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_2920\349334214.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_in